In [1]:
# Import necessary libraries
import pandas as pd
import os
from radon.metrics import mi_visit, h_visit
from radon.complexity import cc_visit
from lizard import analyze_file

# Load the dataset
file_path = 'first_500_characteristics_repo.csv'  # Update this path as needed
df = pd.read_csv(file_path)

# Display the first few rows to understand the structure of your dataset
print("Initial Dataset Preview:")
print(df.head())

# Function to extract source code metrics from a file
def extract_metrics(file_path):
    metrics = {
        'sloc': 0,
        'cyclomatic_complexity': 0,
        'halstead_effort': 0,
        'maintainability_index': 0,
        'nested_block_depth': 0,
    }
    
    try:
        with open(file_path, 'r') as file:
            code = file.read()
            # Cyclomatic Complexity
            cc_results = cc_visit(code)
            metrics['cyclomatic_complexity'] = sum(result.complexity for result in cc_results)
            
            # Halstead Metrics
            halstead_results = h_visit(code)
            metrics['halstead_effort'] = halstead_results.total_effort
            
            # Maintainability Index
            metrics['maintainability_index'] = mi_visit(code, True)
            
            # SLOC (Source Lines of Code)
            metrics['sloc'] = len(code.splitlines())
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

    # Additional metrics using Lizard
    try:
        lizard_analysis = analyze_file(file_path)
        metrics['nested_block_depth'] = max(func.max_nesting_depth for func in lizard_analysis.function_list) if lizard_analysis.function_list else 0
    except Exception as e:
        print(f"Error processing {file_path} with Lizard: {e}")

    return metrics

# Example directory where source code files are located
source_code_directory = 'path_to_source_code'  # Update with the path to the source code related to your repos

# Extracting metrics for each Python file in the directory
all_metrics = []

for root, _, files in os.walk(source_code_directory):
    for file in files:
        if file.endswith('.py'):  # Adjust based on the relevant file types in your repositories
            file_path = os.path.join(root, file)
            file_metrics = extract_metrics(file_path)
            file_metrics['file_path'] = os.path.basename(file_path)  # Simplify the path for merging
            all_metrics.append(file_metrics)

# Convert extracted metrics into a DataFrame
metrics_df = pd.DataFrame(all_metrics)

# Display the extracted metrics
print("Extracted Metrics Preview:")
print(metrics_df.head())

# Save the metrics to a CSV file for further use
metrics_df.to_csv('extracted_metrics.csv', index=False)

# Ensure alignment of merge keys
repository_column = 'full_name'  # Update this to match the identifier in your dataset
metrics_key_column = 'file_path'  # Simplified file path column in metrics_df

# Perform the merge operation
merged_df = pd.merge(df, metrics_df, left_on=repository_column, right_on=metrics_key_column, how='left')

# Check merged DataFrame
print("Merged Dataset Preview:")
print(merged_df.head())

# Save the merged dataset for further processing or model training
merged_df.to_csv('merged_dataset_with_metrics.csv', index=False)

print("Merged dataset with metrics saved successfully.")


Initial Dataset Preview:
                          full_name  is_fork  has_issues           created_at  \
0        josch/cycles_johnson_meyer    False        True  2012-07-04 12:02:27   
1   hantsy/angularjs-cakephp-sample    False        True  2013-11-06 12:56:34   
2          metabase/metabase-deploy    False       False  2015-10-08 00:01:18   
3  sakshamsharma/http-over-protocol    False        True  2016-08-30 10:53:30   
4                  sradley/overflow    False        True  2020-10-27 06:30:05   

                   last_modified            pushed_at main_language  \
0  Sun, 01 Jan 2023 16:35:16 GMT  2018-07-05 03:54:37          Java   
1  Wed, 07 Dec 2022 18:29:56 GMT  2015-10-08 05:00:41           PHP   
2  Sat, 10 Dec 2022 12:07:00 GMT  2022-08-05 22:57:15         Shell   
3  Tue, 15 Nov 2022 20:41:40 GMT  2016-09-25 04:59:55           C++   
4  Thu, 01 Dec 2022 11:13:38 GMT  2022-12-18 03:42:19            Go   

   total_issues_count  open_issues_count  closed_issues_count

KeyError: 'file_path'